# Machine Learning based credit card fraud detection

## Phase 1: Dataset Validation and Graph Feasibility

### Research Direction
Advanced Graph Neural Network (GNN)-based credit card fraud detection.

### Dataset
IEEE-CIS Fraud Detection Dataset.

### Objective
To validate the dataset, understand class imbalance, investigate shared transaction attributes, and evaluate the feasibility of graph-based fraud detection.

### Research Capabilities
- Temporal Graph Learning
- Heterogeneous Graph Neural Networks
- Camouflage-Resistant Graph Learning
- Explainable Graph AI
- Imbalance-Aware Graph Learning

### Expected Outcome
A validated dataset and a documented foundation for constructing a scalable, leakage-aware transaction graph.

## 1. Environment and Configuration

Import the required libraries and define reusable project and dataset paths for consistent execution throughout the notebook.

In [1]:
from pathlib import Path
import pandas as pd
project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

data_dir = project_dir / "data" / "raw"

print("Project directory:", project_dir)
print("Dataset directory:", data_dir)
print("Dataset directory exists:", data_dir.is_dir())

Project directory: c:\Users\rakes\Machine Learning\Minor Project\credit-card-fraud-detection
Dataset directory: c:\Users\rakes\Machine Learning\Minor Project\credit-card-fraud-detection\data\raw
Dataset directory exists: True


## 2. Dataset File Validation

Verify the availability of all required IEEE-CIS Fraud Detection dataset files before performing data analysis.

In [2]:
required_files = [
    "train_transaction.csv",
    "train_identity.csv",
    "test_transaction.csv",
    "test_identity.csv",
    "sample_submission.csv"
]
for filename in required_files:
    file_path = data_dir / filename

    if not file_path.is_file():
        raise FileNotFoundError(f"Missing dataset file: {filename}")
    print(f"{filename}: Available")

print("\nAll required dataset files are available.")

train_transaction.csv: Available
train_identity.csv: Available
test_transaction.csv: Available
test_identity.csv: Available
sample_submission.csv: Available

All required dataset files are available.


## 3. Dataset Loading and Schema Inspection

Load the transaction attributes required for dataset validation, temporal analysis, class imbalance analysis, and graph feasibility investigations.

Selective column loading reduces unnecessary memory consumption and establishes a reusable DataFrame for subsequent analysis.

In [3]:
train_file = data_dir / "train_transaction.csv"

selected_columns = [
    "TransactionID", "isFraud", "TransactionDT",
    "TransactionAmt", "card1", "card2", "card3",
    "card4", "card5", "card6"
]

dataset_columns = pd.read_csv(train_file, nrows=0).columns
transactions = pd.read_csv(train_file, usecols=selected_columns)

print("Total available columns:", len(dataset_columns))
print("Loaded dataset shape:", transactions.shape)

transactions.head()

Total available columns: 394
Loaded dataset shape: (590540, 10)


,TransactionID,isFraud,TransactionDT,TransactionAmt,card1,card2,card3,card4,card5,card6
0,2987000,0,86400,68.5,13926,NaN,150.0,discover,142.0,credit
1,2987001,0,86401,29.0,2755,404.0,150.0,mastercard,102.0,credit
2,2987002,0,86469,59.0,4663,490.0,150.0,visa,166.0,debit
3,2987003,0,86499,50.0,18132,567.0,150.0,mastercard,117.0,debit
4,2987004,0,86506,50.0,4497,514.0,150.0,mastercard,102.0,credit


## 4. Dataset Integrity Validation

Verify transaction identifier uniqueness, missing identifiers, target completeness, and target label validity before further analysis.

These checks establish the reliability of transaction records for supervised machine learning and graph node construction.

In [4]:
integrity_checks = pd.Series({
    "Missing transaction IDs": transactions["TransactionID"].isna().sum(),
    "Duplicate transaction IDs": transactions["TransactionID"].duplicated().sum(),
    "Missing target labels": transactions["isFraud"].isna().sum(),
    "Invalid target labels": (~transactions["isFraud"].isin([0, 1])).sum()
})

print(integrity_checks)

assert integrity_checks.eq(0).all(), "Dataset integrity validation failed."

print("\nDataset integrity validation passed.")

Missing transaction IDs      0
Duplicate transaction IDs    0
Missing target labels        0
Invalid target labels        0
dtype: int64

Dataset integrity validation passed.


### Research Findings

- All 590,540 transactions have unique transaction identifiers.
- No missing or invalid fraud labels were detected.
- All transaction labels belong to the binary classes 0 and 1.
- Unique transaction identifiers support reliable transaction-node mapping during graph construction.

The dataset passed the defined integrity checks and is suitable for subsequent exploratory analysis.

## 5. Data Quality Analysis

Analyze the data types, missing-value counts, and missing-value percentages of the selected transaction attributes.

This investigation supports preprocessing decisions and the selection of suitable attributes for graph construction.

In [5]:
quality_summary = pd.DataFrame({
    "Data Type": transactions.dtypes,
    "Missing Count": transactions.isna().sum(),
    "Missing (%)": (transactions.isna().mean() * 100).round(2)
})

quality_summary

,Data Type,Missing Count,Missing (%)
TransactionID,int64,0,0.00
isFraud,int64,0,0.00
TransactionDT,int64,0,0.00
TransactionAmt,float64,0,0.00
card1,int64,0,0.00
card2,float64,8933,1.51
card3,float64,1565,0.27
card4,str,1577,0.27
card5,float64,4259,0.72
card6,str,1571,0.27


## 6. Class Imbalance Analysis

Analyze the distribution of legitimate and fraudulent transactions and establish a majority-class accuracy baseline.

This investigation demonstrates the importance of imbalance-aware learning and appropriate fraud detection evaluation metrics.

In [6]:
class_counts = transactions["isFraud"].value_counts().sort_index()
class_percentages = class_counts / len(transactions) * 100

class_summary = pd.DataFrame({
    "Count": class_counts,
    "Percentage (%)": class_percentages.round(2)
})

majority_accuracy = class_percentages.max()

print(f"Total transactions: {len(transactions):,}")
print(f"Majority-class accuracy: {majority_accuracy:.2f}%")

class_summary

Total transactions: 590,540
Majority-class accuracy: 96.50%


,Count,Percentage (%)
isFraud,,
0,569877,96.5
1,20663,3.5


## 7. Temporal Validation

Investigate the transaction time range, chronological ordering, missing timestamps, and repeated timestamps.

This analysis establishes the foundation for chronological dataset splitting, temporal graph construction, and future-information leakage prevention.

In [7]:
transaction_time = transactions["TransactionDT"]

time_summary = pd.Series({
    "Missing timestamps": transaction_time.isna().sum(),
    "Negative timestamps": transaction_time.lt(0).sum(),
    "Earliest timestamp": transaction_time.min(),
    "Latest timestamp": transaction_time.max(),
    "Duration (days)": round(
        (transaction_time.max() - transaction_time.min()) / 86400, 2
    ),
    "Chronologically ordered": transaction_time.is_monotonic_increasing,
    "Repeated timestamps": transaction_time.duplicated().sum()
})

time_summary

Missing timestamps                0
Negative timestamps               0
Earliest timestamp            86400
Latest timestamp           15811131
Duration (days)               182.0
Chronologically ordered        True
Repeated timestamps           17191
dtype: object

## 8. Card Attribute Analysis

Analyze the cardinality of all six card attributes and the distribution of card networks.

This investigation supports the selection of suitable transaction features and candidate graph entities for heterogeneous graph construction.

In [8]:
card_columns = ["card1", "card2", "card3", "card4", "card5", "card6"]
card_cardinality = transactions[card_columns].nunique()

card_network_counts = transactions["card4"].value_counts(dropna=False)

card_network_summary = pd.DataFrame({
    "Count": card_network_counts,
    "Percentage (%)": (card_network_counts / len(transactions) * 100).round(2)
})
print("Card attribute cardinality:")
print(card_cardinality)
card_network_summary

Card attribute cardinality:
card1    13553
card2      500
card3      114
card4        4
card5      119
card6        4
dtype: int64


,Count,Percentage (%)
card4,,
visa,384767,65.16
mastercard,189217,32.04
american express,8328,1.41
discover,6651,1.13
NaN,1577,0.27


## 9. Shared Attribute Analysis

Investigate the frequency, repetition, and concentration of card1 values across transactions.

This analysis evaluates the availability of shared-attribute relationships and identifies highly connected groups that may affect graph construction and scalability.

In [9]:
card1_counts = transactions["card1"].value_counts()

repeated_groups = card1_counts[card1_counts > 1]
large_groups = card1_counts[card1_counts > 1000]

shared_summary = pd.Series({
    "Unique card1 values": len(card1_counts),
    "Repeated card1 values": len(repeated_groups),
    "Transactions in repeated groups": repeated_groups.sum(),
    "Repeated transactions (%)": round(
        repeated_groups.sum() / len(transactions) * 100, 2
    ),
    "Groups with over 1000 transactions": len(large_groups),
    "Transactions in large groups": large_groups.sum()
})
print(shared_summary)
print("\nTop 5 card1 groups:")
card1_counts.head(5)

Unique card1 values                    13553.00
Repeated card1 values                  10109.00
Transactions in repeated groups       587096.00
Repeated transactions (%)                 99.42
Groups with over 1000 transactions       104.00
Transactions in large groups          284122.00
dtype: float64

Top 5 card1 groups:


card1
7919     14932
9500     14162
15885    10361
17188    10344
15066     7945
Name: count, dtype: int64

## 10. Graph Scalability Analysis

Compare the theoretical edge requirements of pairwise transaction graphs and transaction-to-attribute bipartite graphs.

The comparison uses card1 groups containing more than 1,000 transactions to evaluate graph construction scalability without explicitly generating graph edges.

In [10]:
pairwise_edges = (large_groups*(large_groups-1)//2).sum()
bipartite_edges = large_groups.sum()
edge_ratio = pairwise_edges/bipartite_edges

scalability_summary = pd.Series({
    "Large attribute groups": len(large_groups),
    "Transactions analyzed": bipartite_edges,
    "Pairwise edges": pairwise_edges,
    "Bipartite edges": bipartite_edges,
    "Edge count ratio": round(edge_ratio, 2)
})
scalability_summary

Large attribute groups    1.040000e+02
Transactions analyzed     2.841220e+05
Pairwise edges            7.054333e+08
Bipartite edges           2.841220e+05
Edge count ratio          2.482850e+03
dtype: float64

## 11. Entity Relationship Analysis

Investigate the relationships between card1 and the remaining card attributes by measuring the number of distinct non-missing attribute values associated with each card1 group.

This analysis supports composite attribute selection and helps identify potentially ambiguous shared-attribute relationships for graph construction.

In [11]:
related_columns = card_columns[1:]
relationship_counts = transactions.groupby("card1")[related_columns].nunique()
relationship_summary = pd.DataFrame({
    "No observed values": relationship_counts.eq(0).sum(),
    "Single value": relationship_counts.eq(1).sum(),
    "Multiple values": relationship_counts.gt(1).sum(),
    "Maximum distinct values": relationship_counts.max()
})
for column in related_columns:
    multi_value_groups = relationship_counts.index[
        relationship_counts[column] > 1
    ]
    relationship_summary.loc[column, "Affected transactions"] = (
        card1_counts.loc[multi_value_groups].sum()
    )
relationship_summary["Affected transactions"] = (
    relationship_summary["Affected transactions"].astype(int)
)
relationship_summary["Affected transactions (%)"] = (
    relationship_summary["Affected transactions"]
    / len(transactions) * 100
).round(2)
relationship_summary

,No observed values,Single value,Multiple values,Maximum distinct values,Affected transactions,Affected transactions (%)
card2,313,12992,248,3,17409,2.95
card3,10,13522,21,2,830,0.14
card4,20,13533,0,1,0,0.00
card5,130,13184,239,3,49177,8.33
card6,11,13439,103,2,5005,0.85


## 12. Research Consistency Validation

Verify that the refactored implementation preserves previously established research findings and maintains internal consistency.

These checks support reproducibility and provide a final validation checkpoint before data preprocessing and model development.

In [12]:
validation_checks = pd.Series({
    "Dataset dimensions preserved": transactions.shape == (590540, 10),
    "Dataset integrity maintained": integrity_checks.eq(0).all(),
    "Class distribution preserved": class_counts.to_dict() == {0: 569877, 1: 20663},
    "Card1 cardinality preserved": len(card1_counts) == 13553,
    "Repeated transactions preserved": repeated_groups.sum() == 587096,
    "Large group count preserved": len(large_groups) == 104,
    "Pairwise edge count preserved": pairwise_edges == 705433284,
    "Bipartite edge count preserved": bipartite_edges == 284122,
    "Relationship groups consistent": (
        relationship_summary[
            ["No observed values", "Single value", "Multiple values"]
        ].sum(axis=1).eq(len(card1_counts)).all()
    ),
    "Card2 findings preserved": (
        relationship_summary.loc["card2", "Affected transactions"] == 17409
    )
})
print(validation_checks.to_string())
assert validation_checks.all(), "Research consistency validation failed."
print("\nAll research consistency checks passed.")

Dataset dimensions preserved       True
Dataset integrity maintained       True
Class distribution preserved       True
Card1 cardinality preserved        True
Repeated transactions preserved    True
Large group count preserved        True
Pairwise edge count preserved      True
Bipartite edge count preserved     True
Relationship groups consistent     True
Card2 findings preserved           True

All research consistency checks passed.


## 13. Phase 1: Research Conclusions

### 13.1 Dataset Validation

- The labeled transaction dataset contains 590,540 transactions and 394 available columns.
- Ten relevant attributes were selected for initial validation and graph feasibility analysis.
- All transaction identifiers are unique.
- No missing or invalid fraud labels were detected.

### 13.2 Class Imbalance

- Legitimate transactions: 569,877 (96.50%).
- Fraudulent transactions: 20,663 (3.50%).
- The majority-class accuracy baseline is 96.50%.

These findings justify the use of imbalance-aware learning and evaluation metrics beyond accuracy.

### 13.3 Temporal Feasibility

- The dataset spans approximately 182 days.
- No missing or negative timestamps were detected.
- Transactions are arranged in non-decreasing chronological order.
- There are 17,191 repeated timestamps beyond their first occurrence.

Chronological evaluation and historical graph construction will be used to prevent future-information leakage.

### 13.4 Shared Attribute Feasibility

- The card1 attribute contains 13,553 distinct values.
- A total of 10,109 card1 values occur more than once.
- Approximately 99.42% of transactions belong to repeated card1 groups.
- A total of 104 groups contain more than 1,000 transactions.

These findings provide structural motivation for investigating shared-attribute graph relationships.

### 13.5 Graph Scalability

For the 104 large card1 groups containing 284,122 transactions:

| Graph representation | Theoretical edges |
| --- | ---: |
| Pairwise transaction graph | 705,433,284 |
| Bipartite transaction-to-attribute graph | 284,122 |

The pairwise representation requires approximately 2,482.85 times as many edges.

A bipartite representation is therefore selected as the initial graph construction approach.

### 13.6 Candidate Entity Selection

The following shared-attribute representations will be investigated:

- card1
- card1 + card2
- card1 + card5

The card1-card2 investigation identified 248 multiple-value groups containing 17,409 transactions.

The card1-card5 investigation identified 239 multiple-value groups containing 49,177 transactions.

These findings support further investigation of composite graph entities.

### 13.7 Research Limitations

- Anonymized card attributes do not establish verified customer or credit card identities.
- Repeated attributes do not guarantee meaningful fraud-related relationships.
- Missing attribute values require careful handling during graph construction.
- The theoretical edge comparison does not measure actual memory usage, execution time, or model performance.
- No machine learning or graph neural network models have been trained during this phase.
- Full-dataset exploratory findings are descriptive and must not be used as held-out performance evidence.

### 13.8 Next Phase

Phase 2 will focus on data preparation, transaction-identity integration, chronological dataset partitioning, and leakage-aware preprocessing.

Graph entity selection and model configurations will be evaluated experimentally without using held-out test results for model selection.